# Choose pilot dates

## Purpose and decision rules

This notebook derives candidate pilot retrieval designs from the frozen 110-scene evaluation pool. The design size is not chosen in advance: it is the smallest set that covers every decade, season and water-level pattern observed within each Landsat mission. Design 1 was approved on 2026-08-08 as the controlled retrieval-candidate set.

The approved 14-scene design includes exactly two of the five scenes linked to LiDAR within ±3 days, leaving three linked scenes unused for independent validation. These are retrieval candidates, not accepted extraction scenes: local cloud, shoreline visibility and Landsat 7 SLC-off effects still require visual QC.

This operationalises `implementation-plan.json` items P020, P023, P024, P025, P042 and P061. No shoreline position, erosion-rate or downstream model result is used in selection.

## Load and validate inputs

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.optimize import Bounds, LinearConstraint, milp

def find_repo_root():
    """Find the repository whether Jupyter started in root or notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "implementation-plan.json").is_file():
            return candidate
    raise FileNotFoundError("Could not locate implementation-plan.json")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from holderness import config

paths = {
    "plan": REPO_ROOT / "implementation-plan.json",
    "scenes": REPO_ROOT / "data/derived/pilot/pilot-water-level-evaluation-scenes.csv",
    "water_levels": REPO_ROOT / "data/derived/pilot/pilot-scene-water-levels.csv",
    "water_summary": REPO_ROOT / "data/derived/pilot/pilot-water-level-summary.json",
    "sectors": REPO_ROOT / "data/interim/pilot/pilot-sector-candidates.csv",
    "lidar": REPO_ROOT / "data/interim/validation/landsat-lidar-pilot-opportunities.csv",
}

print(f"Repository root: {REPO_ROOT}")

Required input checks:

- 110 unique, geometry-eligible scenes;
- one water-level row for each of the three core sectors per scene;
- complete mandatory identifiers; and
- the expected plan items and completed water-level evidence status.

In [ ]:
missing_inputs = [
    str(path.relative_to(REPO_ROOT))
    for path in paths.values()
    if not path.is_file()
]
if missing_inputs:
    raise FileNotFoundError(
        f"Missing required notebook inputs: {missing_inputs}"
    )

tables = {}

for name, path in paths.items():
    if path.suffix == ".csv":
        tables[name] = pd.read_csv(path)

scenes = tables["scenes"]
water_levels = tables["water_levels"]
sectors = tables["sectors"]
lidar = tables["lidar"]

json_inputs = {}

for name, path in paths.items():
    if path.suffix == ".json":
        json_inputs[name] = json.loads(path.read_text())

plan = json_inputs["plan"]
water_summary = json_inputs["water_summary"]

In [ ]:
assert len(scenes) == 110
assert scenes["source_scene_id"].is_unique
assert scenes["primary_geometry_eligible"].eq(True).all()

water_counts = water_levels.groupby("source_scene_id").size()
assert set(water_counts.index) == set(scenes["source_scene_id"])
assert water_counts.eq(3).all()

core_sector_ids = set(
    sectors.loc[
        sectors["role"].eq("core_candidate"),
        "pilot_sector_id",
    ]
)
assert set(water_levels["pilot_sector_id"]) == core_sector_ids

assert water_summary["status"] == (
    "complete_water_level_evidence_dates_still_unselected"
)

In [ ]:
required_ids = {
    "scenes": ["source_scene_id", "source_product_id", "system_index"],
    "water_levels": [
        "source_scene_id",
        "pilot_sector_id",
        "fes_node_id",
        "gtsm_station_id",
    ],
    "sectors": ["pilot_sector_id"],
    "lidar": ["source_scene_id", "pilot_sector_id"],
}

for table_name, id_columns in required_ids.items():
    table = tables[table_name]
    missing_columns = set(id_columns) - set(table.columns)
    assert not missing_columns, (table_name, missing_columns)
    assert table[id_columns].notna().all().all(), table_name

required_plan_ids = {
    "P020", "P023", "P024", "P025", "P042", "P061"
}
plan_ids = {item["id"] for item in plan}
assert required_plan_ids <= plan_ids

## Build one review row per scene

In [ ]:
bands = water_levels.pivot(
    index="source_scene_id",
    columns="pilot_sector_id",
    values="water_level_band",
).add_prefix("water_band__")

lidar_summary = (
    lidar.groupby("source_scene_id", as_index=False)
    .agg(
        lidar_pairs_within_3_days=("lidar_within_3_days", "sum"),
        lidar_pairs_within_7_days=("lidar_within_7_days", "sum"),
    )
)

review = (
    scenes
    .merge(bands, on="source_scene_id", validate="one_to_one")
    .merge(lidar_summary, on="source_scene_id", how="left", validate="one_to_one")
)

lidar_columns = [
    "lidar_pairs_within_3_days",
    "lidar_pairs_within_7_days",
]

review[lidar_columns] = (
    review[lidar_columns]
    .fillna(0)
    .astype(int)
)

In [ ]:
band_columns = list(bands.columns)

review["water_pattern"] = review[band_columns].apply(
    lambda row: (
        row.iloc[0]
        if row.nunique() == 1
        else "mixed_across_core_sectors"
    ),
    axis=1,
)

review["has_middle_band"] = review[band_columns].eq(
    "local_amsl_to_below_plus_0_2_m"
).any(axis=1)

review["consistently_below_amsl"] = review[band_columns].eq(
    "below_local_amsl"
).all(axis=1)

review["consistently_high_water"] = review[band_columns].eq(
    "at_or_above_local_amsl_plus_0_2_m"
).all(axis=1)

In [ ]:
print("===Mission-Decade Counts===")
print(pd.crosstab(review["sensor"], review["decade"]))

print("===Mission-Season Counts===")
print(pd.crosstab(review["sensor"], review["season"]))

print("===Water-Level-Band Counts===")
for col in bands.columns:
    print(bands[col].value_counts())

In [ ]:
review_columns = [
    "source_scene_id",
    "sensor",
    "acquisition_time_utc",
    "decade",
    "season",
    "water_pattern",
    "has_middle_band",
    "cloud_cover_pct",
    "geometric_rmse_m",
    "lidar_pairs_within_3_days",
    "lidar_pairs_within_7_days",
]

display(
    review[review_columns]
    .sort_values(["sensor", "acquisition_time_utc"])
)

## Automated minimum-coverage design

The selection is formulated as a minimum set-cover problem. It must cover every observed mission-specific decade, season and water pattern, include exactly two ±3-day LiDAR-linked dates, and select at most one scene from each full mission–decade–season–water combination.

Among minimum-size solutions, the sum of within-mission geometric-RMSE ranks is minimised first, the sum of within-mission scene-wide cloud ranks second, and scene ID provides a deterministic final tie-break. Lower ranks are better. Scene-wide cloud metadata are only a coarse tie-break and do not replace local visual QC.

In [ ]:
# Prepare selection features

review = review.reset_index(drop=True)

lidar_columns = [
    "lidar_pairs_within_3_days",
    "lidar_pairs_within_7_days",
]
review[lidar_columns] = (
    review[lidar_columns]
    .fillna(0)
    .astype(int)
)

band_columns = list(bands.columns)

review["water_pattern"] = review[band_columns].apply(
    lambda row: (
        row.iloc[0]
        if row.nunique() == 1
        else "mixed"
    ),
    axis=1,
)

review["has_lidar_within_3_days"] = (
    review["lidar_pairs_within_3_days"] > 0
)

review["acquisition_time"] = pd.to_datetime(
    review["acquisition_time_utc"],
    format="mixed",
    utc=True,
)
review["landsat_7_slc_off"] = (
    review["sensor"].eq("L7")
    & review["acquisition_time"].ge(
        pd.Timestamp("2003-06-01", tz="UTC")
    )
)

# Rank quality within each mission so missions with different sensors and
# metadata distributions remain comparable. Integer ranks make the later
# lexicographic optimisation numerically exact.
review["rmse_rank_within_mission"] = (
    review.groupby("sensor")["geometric_rmse_m"]
    .rank(method="min")
    .astype(int)
)

review["cloud_rank_within_mission"] = (
    review.groupby("sensor")["cloud_cover_pct"]
    .rank(method="min")
    .astype(int)
)

# Used only after RMSE and cloud cover are tied.
ordered_scene_ids = {
    scene_id: rank
    for rank, scene_id in enumerate(
        sorted(review["source_scene_id"]),
        start=1,
    )
}

review["scene_id_rank"] = (
    review["source_scene_id"]
    .map(ordered_scene_ids)
    .astype(int)
)

assert review["source_scene_id"].is_unique
assert review["geometric_rmse_m"].notna().all()
assert review["cloud_cover_pct"].notna().all()
assert review["has_lidar_within_3_days"].sum() == 5

display(
    pd.crosstab(
        review["sensor"],
        review["water_pattern"],
    )
)

In [ ]:
# Construct the minimum coverage problem

coverage_inventory = (
    review.groupby("sensor")
    .agg(
        observed_decades=("decade", "nunique"),
        observed_seasons=("season", "nunique"),
        observed_water_patterns=("water_pattern", "nunique"),
    )
)
coverage_inventory["per_mission_lower_bound"] = (
    coverage_inventory.max(axis=1)
)

display(coverage_inventory)

coverage_rows = []
coverage_labels = []

# Require every category observed within each mission.
for mission, mission_rows in review.groupby("sensor", sort=True):
    for dimension in ("decade", "season", "water_pattern"):
        for value in sorted(mission_rows[dimension].unique()):
            coverage_labels.append((mission, dimension, value))

            coverage_rows.append(
                (
                    review["sensor"].eq(mission)
                    & review[dimension].eq(value)
                ).astype(float).to_numpy()
            )

coverage_matrix = np.vstack(coverage_rows)

forced_coverage_rows = []
for label, row in zip(coverage_labels, coverage_matrix):
    candidate_indices = np.flatnonzero(row)
    if len(candidate_indices) == 1:
        candidate = review.iloc[candidate_indices[0]]
        forced_coverage_rows.append({
            "sensor": label[0],
            "coverage_dimension": label[1],
            "coverage_value": label[2],
            "source_scene_id": candidate["source_scene_id"],
            "cloud_cover_pct": candidate["cloud_cover_pct"],
            "geometric_rmse_m": candidate["geometric_rmse_m"],
        })

forced_coverage_scenes = pd.DataFrame(forced_coverage_rows)
display(
    Markdown(
        "**Scenes forced by a category with only one candidate:**"
    )
)
display(forced_coverage_scenes)

# Prevent duplicate selections from the same complete joint stratum. This
# maximises the number of distinct combinations represented by the design.
joint_dimensions = [
    "sensor",
    "decade",
    "season",
    "water_pattern",
]

joint_rows = []

for _, indices in review.groupby(
    joint_dimensions,
    sort=True,
).groups.items():
    row = np.zeros(len(review), dtype=float)
    row[np.asarray(list(indices), dtype=int)] = 1.0
    joint_rows.append(row)

joint_matrix = np.vstack(joint_rows)

ones = np.ones(len(review), dtype=float)

lidar_indicator = (
    review["has_lidar_within_3_days"]
    .astype(float)
    .to_numpy()
)

integrality = np.ones(len(review), dtype=int)
bounds = Bounds(0, 1)

base_constraints = [
    # Every observed mission-specific category must be represented.
    LinearConstraint(
        coverage_matrix,
        lb=np.ones(len(coverage_rows)),
        ub=np.full(len(coverage_rows), np.inf),
    ),

    # Use two of the five LiDAR-linked dates, leaving three unused.
    LinearConstraint(
        lidar_indicator[None, :],
        lb=np.array([2.0]),
        ub=np.array([2.0]),
    ),

    # At most one scene from any full joint stratum.
    LinearConstraint(
        joint_matrix,
        lb=np.full(len(joint_rows), -np.inf),
        ub=np.ones(len(joint_rows)),
    ),
]


def run_milp(objective, constraints):
    """Run one binary scene-selection optimisation."""

    result = milp(
        c=np.asarray(objective, dtype=float),
        integrality=integrality,
        bounds=bounds,
        constraints=constraints,
        options={"disp": False},
    )

    if not result.success:
        raise RuntimeError(result.message)

    return result


minimum_result = run_milp(
    objective=ones,
    constraints=base_constraints,
)

minimum_scene_count = int(round(minimum_result.fun))

coverage_lower_bound = int(
    coverage_inventory["per_mission_lower_bound"].sum()
)
assert coverage_lower_bound == 14
assert minimum_scene_count == coverage_lower_bound

print(
    "Minimum scene count satisfying all coverage requirements:",
    minimum_scene_count,
)

In [ ]:
# Generate five defensible alternatives

def solve_design(excluded_designs=()):
    """Return the best remaining minimum-size design."""

    constraints = list(base_constraints)

    # Fix the solution at the proven minimum size.
    constraints.append(
        LinearConstraint(
            ones[None, :],
            lb=np.array([minimum_scene_count], dtype=float),
            ub=np.array([minimum_scene_count], dtype=float),
        )
    )

    # Exclude complete designs already returned.
    for excluded_indices in excluded_designs:
        exclusion = np.zeros(len(review), dtype=float)
        exclusion[list(excluded_indices)] = 1.0

        constraints.append(
            LinearConstraint(
                exclusion[None, :],
                lb=np.array([-np.inf]),
                ub=np.array([minimum_scene_count - 1.0]),
            )
        )

    # First tie-break: lowest within-mission geometric RMSE ranks.
    rmse_objective = review[
        "rmse_rank_within_mission"
    ].to_numpy(dtype=float)

    rmse_result = run_milp(rmse_objective, constraints)
    best_rmse = int(round(rmse_result.fun))

    constraints.append(
        LinearConstraint(
            rmse_objective[None, :],
            lb=np.array([best_rmse], dtype=float),
            ub=np.array([best_rmse], dtype=float),
        )
    )

    # Second tie-break: lowest scene-wide cloud-cover ranks.
    cloud_objective = review[
        "cloud_rank_within_mission"
    ].to_numpy(dtype=float)

    cloud_result = run_milp(cloud_objective, constraints)
    best_cloud = int(round(cloud_result.fun))

    constraints.append(
        LinearConstraint(
            cloud_objective[None, :],
            lb=np.array([best_cloud], dtype=float),
            ub=np.array([best_cloud], dtype=float),
        )
    )

    # Final deterministic tie-break.
    id_objective = review["scene_id_rank"].to_numpy(dtype=float)
    final_result = run_milp(id_objective, constraints)

    return tuple(np.flatnonzero(final_result.x > 0.5))


design_indices = []

for _ in range(5):
    design_indices.append(
        solve_design(excluded_designs=design_indices)
    )

assert len(design_indices) == 5
assert len(set(design_indices)) == 5

In [ ]:
# Inspect alternatives

alternative_summaries = []
alternative_members = []

for design_id, indices in enumerate(design_indices, start=1):
    design = review.iloc[list(indices)].copy()
    design["design_id"] = design_id

    water_counts = design["water_pattern"].value_counts()

    alternative_summaries.append({
        "design_id": design_id,
        "scene_count": len(design),
        "L5": int(design["sensor"].eq("L5").sum()),
        "L7": int(design["sensor"].eq("L7").sum()),
        "L8": int(design["sensor"].eq("L8").sum()),
        "L9": int(design["sensor"].eq("L9").sum()),
        "below_amsl": int(
            water_counts.get("below_local_amsl", 0)
        ),
        "at_or_above_plus_0_2_m": int(
            water_counts.get(
                "at_or_above_local_amsl_plus_0_2_m",
                0,
            )
        ),
        "mixed": int(water_counts.get("mixed", 0)),
        "lidar_linked_scenes": int(
            design["has_lidar_within_3_days"].sum()
        ),
        "L7_slc_off_scenes": int(
            design["landsat_7_slc_off"].sum()
        ),
        "rmse_rank_sum": int(
            design["rmse_rank_within_mission"].sum()
        ),
        "cloud_rank_sum": int(
            design["cloud_rank_within_mission"].sum()
        ),
        "mean_geometric_rmse_m": float(
            design["geometric_rmse_m"].mean()
        ),
        "mean_scene_cloud_pct": float(
            design["cloud_cover_pct"].mean()
        ),
        "max_scene_cloud_pct": float(
            design["cloud_cover_pct"].max()
        ),
    })

    alternative_members.append(design)

alternative_summary = pd.DataFrame(alternative_summaries)

alternative_member_table = pd.concat(
    alternative_members,
    ignore_index=True,
)

display(
    alternative_summary.round(3)
    .sort_values("design_id")
)

In [ ]:
# Freeze the approved retrieval-candidate design

CHOSEN_DESIGN_ID = config.PILOT_RETRIEVAL_DESIGN_ID
assert CHOSEN_DESIGN_ID == 1

selected_indices = design_indices[CHOSEN_DESIGN_ID - 1]
selected = review.iloc[list(selected_indices)].copy()

approved_scene_ids = set(config.PILOT_RETRIEVAL_SCENE_IDS)
assert len(approved_scene_ids) == minimum_scene_count
assert set(selected["source_scene_id"]) == approved_scene_ids
assert selected["source_scene_id"].is_unique
assert selected["primary_geometry_eligible"].eq(True).all()
assert not selected.duplicated(joint_dimensions).any()

for mission, dimension, value in coverage_labels:
    covered = (
        selected["sensor"].eq(mission)
        & selected[dimension].eq(value)
    )
    assert covered.any(), (mission, dimension, value)

mission_counts = (
    selected["sensor"]
    .value_counts()
    .sort_index()
    .to_dict()
)
assert mission_counts == {
    "L5": 3,
    "L7": 4,
    "L8": 4,
    "L9": 3,
}

selected_lidar_scene_ids = sorted(
    selected.loc[
        selected["has_lidar_within_3_days"],
        "source_scene_id",
    ]
)
all_lidar_scene_ids = set(
    review.loc[
        review["has_lidar_within_3_days"],
        "source_scene_id",
    ]
)
reserved_lidar_scene_ids = sorted(
    all_lidar_scene_ids - set(selected_lidar_scene_ids)
)
assert len(selected_lidar_scene_ids) == 2
assert len(reserved_lidar_scene_ids) == 3
assert selected["landsat_7_slc_off"].sum() == 2

forced_scene_ids = set(
    forced_coverage_scenes["source_scene_id"]
)
assert forced_scene_ids <= approved_scene_ids

display(
    Markdown(
        "### Approved retrieval design\n\n"
        "Design 1 was approved on 2026-08-08. It preserves the "
        "minimum 14-scene coverage design and the declared RMSE-first, "
        "cloud-second ranking. Approval authorises manifest creation, "
        "not shoreline extraction."
    )
)

review_columns = [
    "source_scene_id",
    "sensor",
    "acquisition_time_utc",
    "decade",
    "season",
    "water_pattern",
    *band_columns,
    "cloud_cover_pct",
    "geometric_rmse_m",
    "has_lidar_within_3_days",
    "landsat_7_slc_off",
]
display(
    selected[review_columns]
    .sort_values(["sensor", "acquisition_time_utc"])
)

coverage_requirements = [
    {
        "sensor": str(mission),
        "dimension": str(dimension),
        "value": str(value),
    }
    for mission, dimension, value in coverage_labels
]

manifest_scene_columns = [
    "source_scene_id",
    "source_product_id",
    "system_index",
    "sensor",
    "acquisition_time_utc",
    "gee_collection",
    "collection_version",
    "wrs_path",
    "wrs_row",
    "decade",
    "season",
    "water_pattern",
    *band_columns,
    "cloud_cover_pct",
    "geometric_rmse_m",
    "primary_geometry_eligible",
    "covered_sector_ids",
    "has_lidar_within_3_days",
    "landsat_7_slc_off",
    "rmse_rank_within_mission",
    "cloud_rank_within_mission",
]
manifest_scenes = json.loads(
    selected[manifest_scene_columns]
    .sort_values(["sensor", "acquisition_time_utc"])
    .to_json(orient="records")
)

manifest = {
    "schema_version": 1,
    "status": "approved_for_controlled_retrieval_and_visual_qc",
    "decision_date": "2026-08-08",
    "design_id": CHOSEN_DESIGN_ID,
    "scene_count": len(selected),
    "decision_rationale": [
        (
            "Design 1 is the optimiser-preferred minimum design under "
            "the approved lexicographic ranking: within-mission "
            "geometric-RMSE rank, then scene-wide cloud rank, then "
            "deterministic source-scene ID."
        ),
        (
            "The design retains every mission-specific observed decade, "
            "season and water-level pattern and uses two of five ±3-day "
            "LiDAR-linked scenes, reserving three for validation."
        ),
        (
            "The forced L9 high-water scene is retained despite 89.17% "
            "scene-wide cloud because local Holderness conditions are "
            "unknown until retrieval and visual QC."
        ),
    ],
    "selection_rules": {
        "minimum_scene_count": minimum_scene_count,
        "mission_counts": mission_counts,
        "coverage_requirements": coverage_requirements,
        "maximum_per_joint_stratum": 1,
        "selected_lidar_linked_scene_count": 2,
        "ranking": [
            "within_mission_geometric_rmse_rank",
            "within_mission_scene_wide_cloud_rank",
            "source_scene_id",
        ],
    },
    "validation_checks": {
        "exact_approved_scene_ids": True,
        "unique_scene_ids": True,
        "all_primary_geometry_eligible": True,
        "all_mission_specific_categories_covered": True,
        "no_duplicate_joint_strata": True,
        "mission_counts_match": True,
        "two_lidar_linked_scenes_selected": True,
        "three_lidar_linked_scenes_reserved": True,
        "forced_single_candidate_categories_retained": True,
    },
    "lidar_allocation": {
        "selected_scene_ids": selected_lidar_scene_ids,
        "reserved_scene_ids": reserved_lidar_scene_ids,
    },
    "forced_single_candidate_categories": json.loads(
        forced_coverage_scenes.to_json(orient="records")
    ),
    "visual_qc_warnings": [
        (
            "LC09_202022_20220607 has 89.17% provider scene-wide cloud "
            "and is forced by L9 high-water coverage."
        ),
        (
            "The two selected post-2003 Landsat 7 scenes are SLC-off "
            "and require local gap inspection."
        ),
        (
            "The four comparison designs are not universal per-scene "
            "fallbacks because 11 scenes are shared by all five designs."
        ),
    ],
    "not_authorised": [
        "final scene acceptance",
        "shoreline extraction",
    ],
    "inputs": {
        name: {
            "path": str(path.relative_to(REPO_ROOT)),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for name, path in paths.items()
    },
    "scenes": manifest_scenes,
}

manifest_path = config.PILOT_RETRIEVAL_MANIFEST
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n"
)
print(f"Wrote {manifest_path.relative_to(REPO_ROOT)}")

Design 1 is now frozen as a controlled retrieval-candidate set. The next practical stage is imagery retrieval followed by local visual QC of cloud, shoreline visibility, masks and Landsat 7 SLC-off gaps. A failed forced scene has no replacement inside the frozen pool without revisiting the coverage design. Shoreline extraction remains out of scope.